# Simple Vectorless RAG
**Reasoning-based RAG — No Vector DB, No Chunking**

Uses only: `groq` (free) + `PyPDF2` + `requests` — no PageIndex API needed.

## Step 0 — Install

In [1]:
!pip install -q groq PyPDF2 python-dotenv requests


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 0.1 — Setup

In [2]:
import os, json, requests
import PyPDF2
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
GROQ_MODEL   = "llama-3.3-70b-versatile"
client       = Groq(api_key=GROQ_API_KEY)

print("Groq:", "OK" if GROQ_API_KEY else "MISSING — add GROQ_API_KEY to .env")

def call_llm(prompt, response_json=False):
    kwargs = {"response_format": {"type": "json_object"}} if response_json else {}
    res = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        **kwargs
    )
    return res.choices[0].message.content.strip()

Groq: OK


## Step 1 — Load PDF\nUsing the **2023 Federal Reserve Annual Report** (local file).

In [3]:
PDF_PATH = "./Docs/2023-annual-report-truncated.pdf"   # local PDF, no download needed

reader = PyPDF2.PdfReader(PDF_PATH)
pages  = [reader.pages[i].extract_text() or "" for i in range(len(reader.pages))]
print(f"PDF loaded: {len(pages)} pages")

PDF loaded: 50 pages


## Step 1.1 — Build Tree Structure
LLM reads the document and builds a hierarchical tree (like a Table of Contents).

In [4]:
def build_tree(pages):
    # Strip non-ASCII (math symbols etc.) to avoid breaking JSON mode
    def clean(text):
        return text.encode("ascii", "ignore").decode("ascii")

    sample = ""
    for i, text in enumerate(pages[:12], 1):
        sample += f"\n--- Page {i} ---\n{clean(text)[:500]}\n"

    prompt = f"""Analyze this document and extract its section structure as a tree.

Document sample:
{sample}

Return a JSON object with key "tree" containing an array of sections.
Each section must have:
- node_id: string like "0001"
- title: section name string
- page_index: integer page number
- summary: one sentence string
- nodes: array of child sections (empty array if none)

Example: {{"tree": [{{"node_id": "0001", "title": "Abstract", "page_index": 1, "summary": "Overview.", "nodes": []}}]}}"""

    raw    = call_llm(prompt, response_json=True)
    result = json.loads(raw)
    if "tree" in result:
        return result["tree"]
    for v in result.values():
        if isinstance(v, list):
            return v
    return []

print("Building tree... (~10 seconds)")
tree = build_tree(pages)

def print_tree(nodes, indent=0):
    for n in nodes:
        print("  " * indent + f"[{n['node_id']}] {n['title']}  (p.{n.get('page_index','?')})")
        if n.get("nodes"):
            print_tree(n["nodes"], indent + 1)

print(f"Tree built: {len(tree)} top-level sections\n")
print("Document Structure:")
print_tree(tree)

Building tree... (~10 seconds)


Tree built: 1 top-level sections

Document Structure:
[0001] REPORT TO CONGRESS  (p.1)
  [0002] About the Federal Reserve  (p.5)
  [0003] Contents  (p.3)
    [0004] 1. Overview  (p.7)
    [0005] 2. Monetary Policy and Economic Developments  (p.9)
      [0006] Recent Economic and Financial Developments  (p.10)
      [0007] Economic Activity  (p.11)
      [0008] Financial Stability  (p.12)


## Step 2 — Reasoning-Based Retrieval (Tree Search)
LLM reads the tree and picks which nodes likely contain the answer.

In [5]:
query = "What are the conclusions in this document?"

def compress_tree(nodes):
    return [{"node_id": n["node_id"], "title": n["title"],
             "page": n.get("page_index", "?"),
             "summary": n.get("summary", ""),
             **({"nodes": compress_tree(n["nodes"])} if n.get("nodes") else {})}
            for n in nodes]

search_prompt = f"""You are given a question and a document tree (Table of Contents with summaries).
Find all node_ids likely to contain the answer.

Question: {query}

Document tree:
{json.dumps(compress_tree(tree), indent=2)}

Reply ONLY in JSON:
{{"thinking": "<your reasoning>", "node_list": ["0001", "0002"]}}"""

result   = json.loads(call_llm(search_prompt, response_json=True))
node_ids = result["node_list"]

print("Reasoning Process:")
print(result["thinking"])
print("\nRetrieved Node IDs:", node_ids)

Reasoning Process:
To find the conclusions in the document, we need to look for sections that summarize or wrap up the main points. The question is asking for conclusions, which are typically found at the end of a report or document. However, the provided document tree does not explicitly show a 'Conclusion' section. Given the structure, the conclusions might be inferred from the last sections of the major divisions of the report. Since '0005' discusses 'Monetary Policy and Economic Developments' and has subsections, and there isn't a clear conclusion section, we might infer that conclusions could be drawn from the summaries of major sections or the final parts of significant divisions. Without explicit conclusion headings, we consider sections that could logically contain summary or concluding remarks, such as the last parts of '0005' or any potential summaries within '0001'.

Retrieved Node IDs: ['0005', '0008']


## Step 2.1 — Show Retrieved Nodes

In [6]:
def find_nodes(tree, target_ids):
    found = []
    for n in tree:
        if n["node_id"] in target_ids:
            found.append(n)
        if n.get("nodes"):
            found.extend(find_nodes(n["nodes"], target_ids))
    return found

retrieved = find_nodes(tree, node_ids)
print("Retrieved Nodes:\n")
for n in retrieved:
    print(f"  Node ID: {n['node_id']}   Page: {n.get('page_index','?')}   Title: {n['title']}")

Retrieved Nodes:

  Node ID: 0005   Page: 9   Title: 2. Monetary Policy and Economic Developments
  Node ID: 0008   Page: 12   Title: Financial Stability


## Step 3 — Extract Context from PDF Pages

In [7]:
def get_context(nodes, pages):
    parts = []
    for n in nodes:
        pg = n.get("page_index", 1)
        text = ""
        for p in range(max(0, pg - 1), min(len(pages), pg + 2)):
            text += pages[p]
        parts.append(f"[{n['title']} | Page {pg}]\n{text[:2000]}")
    return "\n\n---\n\n".join(parts)

context = get_context(retrieved, pages)
print("Retrieved Context (first 500 chars):\n")
print(context[:500], "...")

Retrieved Context (first 500 chars):

[2. Monetary Policy and Economic Developments | Page 9]
2Monetar y Policy and Economic
Developments
The Federal Reser ve conducts the nation’ s monetar y policy to promote maximum emplo yment, stable
prices, and moderate long-ter m interest rates in the U.S. econom y. This section re views U.S. monetar y
policy and economic de velopments in 2023 b y providing excer pts and select figures from the Monetar y
Policy Repor tpublished in March 2024 and June 2023 .1The repor t, submitted semiannually  ...


## Step 3.1 — Generate Answer

In [8]:
answer_prompt = f"""Answer the question using ONLY the context below. Be clear and concise.

Question: {query}

Context:
{context}

Answer:"""

answer = call_llm(answer_prompt)
print("Generated Answer:\n")
print(answer)

Generated Answer:

The conclusions in this document are:

1. Inflation has eased substantially over the past year, but remains above the Federal Open Market Committee's (FOMC) objective of 2 percent.
2. The labor market remains relatively tight, with low unemployment and elevated job vacancies.
3. The banking system remains sound and resilient, but areas of risk warrant continued monitoring.
4. The FOMC is committed to returning inflation to its 2 percent objective and remains attentive to inflation risks.
5. Financial stability is maintained, but vulnerabilities from financial-sector leverage and declines in asset values are notable.
